# MH-Weed16: Real-Time Multiclass Weed Detection with YOLO11 and SAHI

An end-to-end computer vision pipeline for agricultural weed detection using the MH-Weed16 benchmark dataset and Ultralytics YOLO11 Nano.

### Architectural Highlights
- **Direct Raw Data Processing**: Operates directly on native 1920x1080 Full HD captures without downsampling or offline image duplication.
- **Slicing Aided Hyper Inference (SAHI)**: On-the-fly sliding-window patch evaluation at native resolution, preventing resolution loss and blur for small weed seedlings.
- **Intel Arc XPU Acceleration**: Automatic Mixed Precision (amp=True) float16 training with AdamW optimizer and rectangular batching.
- **Cleaned Dataset Topology**: Purged degenerate zero-dimension annotations and aligned 15 distinct weed classes.

## 1. Setup and Environment

In [ ]:
import concurrent.futures
import hashlib
import os
import pathlib
import random
import time
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import torch
from ultralytics import YOLO
import yaml
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# Deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Detect compute device
if hasattr(torch, 'xpu') and torch.xpu.is_available():
    device = 'xpu'
    device_name = torch.xpu.get_device_name(0)
elif torch.cuda.is_available():
    device = '0'
    device_name = torch.cuda.get_device_name(0)
else:
    device = 'cpu'
    device_name = 'CPU'

print("=" * 65)
print(f"PyTorch Version:  {torch.__version__}")
print(f"Compute Device:   {device} ({device_name})")
print("=" * 65)

ROOT_DIR = pathlib.Path(".").resolve()


## 2. Dataset Preparation and Directory Configuration

The pipeline points directly to the pristine raw MH-Weed16 captures without creating duplicate resized images on disk.
- Images: Intel RealSense Depth_Clicks (1920x1080 Full HD).
- Annotations: YOLO Darknet format.
- Splits: 80% Train (5,324), 10% Validation (665), 10% Test (667) partitioned at parent image level.

In [ ]:
RAW_IMAGES = ROOT_DIR / "Crop with Weeds" / "intel Real Sense Depth_Clicks" / "intel Real Sense Depth_Clicks"
RAW_LABELS = ROOT_DIR / "Crop with Weeds" / "intel Real Sense Depth_Annotations" / "intel Real Sense Depth_Annotations" / "YOLO_darknet"

IMAGES_DIR = ROOT_DIR / "images"
LABELS_DIR = ROOT_DIR / "labels"

# Create symbolic junctions if they do not exist
if not IMAGES_DIR.exists():
    os.system(f'cmd /c mklink /J "{IMAGES_DIR}" "{RAW_IMAGES}"')
if not LABELS_DIR.exists():
    os.system(f'cmd /c mklink /J "{LABELS_DIR}" "{RAW_LABELS}"')

raw_image_files = sorted([p for p in IMAGES_DIR.glob('*.*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f"Raw images discovered: {len(raw_image_files):,}")

# 80 / 10 / 10 Train / Val / Test Split
shuffled_images = list(raw_image_files)
random.seed(SEED)
random.shuffle(shuffled_images)

n_total = len(shuffled_images)
n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)

train_paths = [os.path.normpath(str(p)) for p in shuffled_images[:n_train]]
val_paths = [os.path.normpath(str(p)) for p in shuffled_images[n_train:n_train + n_val]]
test_paths = [os.path.normpath(str(p)) for p in shuffled_images[n_train + n_val:]]

def write_split_file(filepath, paths):
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(paths) + '\n')

write_split_file(ROOT_DIR / "train.txt", train_paths)
write_split_file(ROOT_DIR / "val.txt", val_paths)
write_split_file(ROOT_DIR / "test.txt", test_paths)

print(f"Splits saved: Train={len(train_paths):,} | Val={len(val_paths):,} | Test={len(test_paths):,}")

# Dataset YAML configuration (15 active weed species in detection set)
CLASS_NAMES = {
    0: "kena",
    1: "lavhala",
    2: "lambs_quarters",
    3: "little_mallow",
    4: "moti_dudhi",
    5: "obscure_morning_glory",
    6: "asian_pigeonwings",
    7: "bilayat",
    8: "choti_dudhi",
    9: "digitaria",
    10: "gajar_gavat",
    11: "graceful_sandmat",
    12: "sicklepod",
    13: "harali",
    14: "dwarf_cassia"
}

yaml_content = {
    'path': str(ROOT_DIR).replace('\\', '/'),
    'train': 'train.txt',
    'val': 'val.txt',
    'test': 'test.txt',
    'names': CLASS_NAMES
}

yaml_file = ROOT_DIR / "mhweed16.yaml"
with open(yaml_file, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"Updated dataset YAML: {yaml_file}")


## 3. Exploratory Data Analysis (EDA) and Dataset Topology

In [ ]:
ann_files = sorted(list(LABELS_DIR.glob('*.txt')))
box_records = []
class_counts = Counter()
boxes_per_image = []

for ann_path in ann_files:
    with open(ann_path, 'r', encoding='utf-8') as f:
        lines = [line.strip().split() for line in f if line.strip()]
    
    boxes_per_image.append(len(lines))
    for parts in lines:
        cls_id = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        class_counts[cls_id] += 1
        box_records.append({
            'image_stem': ann_path.stem,
            'class_id': cls_id,
            'name': CLASS_NAMES.get(cls_id, str(cls_id)),
            'xc': xc,
            'yc': yc,
            'width': w,
            'height': h,
            'aspect_ratio': w / max(h, 1e-6),
            'area': w * h,
            'perimeter': 2.0 * (w + h),
            'center_dist': np.sqrt((xc - 0.5)**2 + (yc - 0.5)**2)
        })

df_boxes = pd.DataFrame(box_records)
print(f"Total Bounding Box Instances: {len(df_boxes):,}")
print(f"Total Labeled Images:         {len(ann_files):,}")
print(f"Unique Weed Species Present:  {len(class_counts)}")
print(f"Mean Boxes Per Image:         {np.mean(boxes_per_image):.2f}")
print(f"Max Boxes in a Single Image:  {np.max(boxes_per_image)}")

# Instance distribution and density plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if i in class_counts]
sns.countplot(data=df_boxes, x='name', order=order, ax=axes[0], color='#2c7bb6')
axes[0].set_title("Instance Frequency per Weed Species", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Weed Species")
axes[0].set_ylabel("Instance Count")
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

sns.histplot(boxes_per_image, bins=range(0, 35), ax=axes[1], color='#1a9641', kde=False)
axes[1].set_title("Bounding Box Density per Image Frame", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Number of Weed Bounding Boxes")
axes[1].set_ylabel("Image Count")
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# Spatial Heatmap and Bounding Box Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.kdeplot(
    data=df_boxes.sample(min(10000, len(df_boxes)), random_state=SEED),
    x='xc',
    y='yc',
    cmap='YlOrRd',
    fill=True,
    thresh=0.02,
    levels=15,
    ax=axes[0]
)
axes[0].set_title("Spatial Distribution of Weed Centers (1080p FOV)", fontsize=11, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(1, 0)
axes[0].set_aspect('equal')

sns.histplot(df_boxes['width'], bins=35, kde=True, ax=axes[1], color='#2c7bb6')
axes[1].set_title("Normalized Box Width Distribution", fontsize=11, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

sns.histplot(df_boxes['height'], bins=35, kde=True, ax=axes[2], color='#d7191c')
axes[2].set_title("Normalized Box Height Distribution", fontsize=11, fontweight='bold')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# Ground Truth Bounding Box Alignment on Raw 1080p Frame
def show_sample_ground_truth(img_path, ann_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if ann_path.exists():
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                name = CLASS_NAMES.get(cls_id, str(cls_id))
                cv2.putText(img, name, (x1, max(22, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.title(f"Raw 1080p Sample ({w}x{h}): {img_path.name} | Ground Truth Bounding Boxes", fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

sample_img = raw_image_files[0]
sample_ann = LABELS_DIR / f"{sample_img.stem}.txt"
show_sample_ground_truth(sample_img, sample_ann)


## 4. Multi-Modal Feature Engineering and Spectral Indices
Extract color, vegetation index, and texture descriptors across weed instances:
- Vegetation Indices: Excess Green ($2G - R - B$), Excess Red ($1.4R - G$), and Normalized Difference ($NDI$).
- Color spaces: Channel means in RGB and HSV.
- Contrast and Texture: Laplacian variance (edge sharpness) and intensity standard deviation.

In [ ]:
def extract_instance_features(img_path):
    ann_path = LABELS_DIR / f"{img_path.stem}.txt"
    if not ann_path.exists():
        return []
    
    img = cv2.imread(str(img_path))
    if img is None:
        return []
    
    h_img, w_img = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        lines = [line.strip().split() for line in f if line.strip()]
    
    n_boxes = len(lines)
    records = []
    
    for parts in lines:
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        
        x1 = max(0, int((xc - bw / 2.0) * w_img))
        y1 = max(0, int((yc - bh / 2.0) * h_img))
        x2 = min(w_img, int((xc + bw / 2.0) * w_img))
        y2 = min(h_img, int((yc + bh / 2.0) * h_img))
        
        patch_rgb = img_rgb[y1:y2, x1:x2]
        if patch_rgb.size < 12:
            continue
        
        patch_hsv = img_hsv[y1:y2, x1:x2]
        patch_gray = img_gray[y1:y2, x1:x2]
        
        rf = patch_rgb[:, :, 0].astype(np.float32) / 255.0
        gf = patch_rgb[:, :, 1].astype(np.float32) / 255.0
        bf = patch_rgb[:, :, 2].astype(np.float32) / 255.0
        
        exg = 2.0 * gf - rf - bf
        exr = 1.4 * rf - gf
        denom = gf + rf + 1e-6
        ndi = (gf - rf) / denom
        
        lap_var = float(cv2.Laplacian(patch_gray, cv2.CV_64F).var()) if patch_gray.size >= 16 else 0.0
        
        records.append({
            'image_name': img_path.name,
            'class_id': cls_id,
            'species': CLASS_NAMES.get(cls_id, str(cls_id)),
            'xc': xc,
            'yc': yc,
            'width': bw,
            'height': bh,
            'aspect_ratio': bw / max(bh, 1e-6),
            'area': bw * bh,
            'perimeter': 2.0 * (bw + bh),
            'center_dist': np.sqrt((xc - 0.5)**2 + (yc - 0.5)**2),
            'boxes_in_frame': n_boxes,
            'r_mean': float(np.mean(rf)),
            'g_mean': float(np.mean(gf)),
            'b_mean': float(np.mean(bf)),
            'hue_mean': float(np.mean(patch_hsv[:, :, 0])),
            'sat_mean': float(np.mean(patch_hsv[:, :, 1])),
            'val_mean': float(np.mean(patch_hsv[:, :, 2])),
            'exg_mean': float(np.mean(exg)),
            'exr_mean': float(np.mean(exr)),
            'ndi_mean': float(np.mean(ndi)),
            'contrast_std': float(np.std(patch_gray)),
            'laplacian_var': lap_var
        })
        
    return records

sample_feature_images = raw_image_files[:min(1000, len(raw_image_files))]
print(f"Extracting multi-modal features across sample of {len(sample_feature_images):,} raw frames...")
t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    extracted_batches = list(executor.map(extract_instance_features, sample_feature_images))

feature_list = [r for batch in extracted_batches for r in batch]
df_features = pd.DataFrame(feature_list)
t1 = time.time()

print(f"Extraction complete in {t1 - t0:.2f}s: Extracted {len(df_features):,} weed instances.")
df_features.head()


## 5. Principal Component Analysis (PCA) and Dimensionality Reduction

In [ ]:
feature_cols = [
    'width', 'height', 'aspect_ratio', 'area', 'perimeter', 'center_dist',
    'r_mean', 'g_mean', 'b_mean', 'hue_mean', 'sat_mean', 'val_mean',
    'exg_mean', 'exr_mean', 'ndi_mean', 'contrast_std', 'laplacian_var'
]

X = df_features[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
df_features['pca_1'] = X_pca[:, 0]
df_features['pca_2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
top_species = df_features['species'].value_counts().head(8).index
sns.scatterplot(
    data=df_features[df_features['species'].isin(top_species)],
    x='pca_1',
    y='pca_2',
    hue='species',
    alpha=0.6,
    s=25
)
plt.title(f"PCA Projection of Weed Features (Explained Var: {pca.explained_variance_ratio_.sum()*100:.1f}%)", fontsize=12, fontweight='bold')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 6. Outlier Detection and Anomaly Auditing

In [ ]:
iso_forest = IsolationForest(contamination=0.01, random_state=SEED, n_jobs=-1)
df_features['anomaly_score'] = -iso_forest.fit(X_scaled).score_samples(X_scaled)
df_features['outlier_iso'] = iso_forest.predict(X_scaled) == -1

print("=" * 55)
print("             DATASET OUTLIER AUDIT SUMMARY            ")
print("=" * 55)
print(f"Total Instances Analyzed:  {len(df_features):,}")
print(f"Isolation Forest Outliers: {df_features['outlier_iso'].sum():,} ({df_features['outlier_iso'].mean()*100:.2f}%)")
print("=" * 55)


## 7. Agricultural Contrast Enhancement (CLAHE and ExG)

In [ ]:
def apply_clahe(img_rgb, clip_limit=2.0):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)

def compute_exg(img_rgb):
    f = img_rgb.astype(np.float32) / 255.0
    r, g, b = f[:, :, 0], f[:, :, 1], f[:, :, 2]
    return 2.0 * g - r - b

raw_sample = cv2.cvtColor(cv2.imread(str(raw_image_files[0])), cv2.COLOR_BGR2RGB)
clahe_sample = apply_clahe(raw_sample)
exg_sample = compute_exg(raw_sample)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(raw_sample)
axes[0].set_title("Raw 1080p RGB", fontsize=11)
axes[0].axis('off')

axes[1].imshow(clahe_sample)
axes[1].set_title("CLAHE Contrast Enhanced", fontsize=11)
axes[1].axis('off')

im3 = axes[2].imshow(exg_sample, cmap='YlGn')
axes[2].set_title("Excess Green Index (ExG)", fontsize=11)
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 8. Model Training (YOLO11 Nano End-to-End)

Train the detector end-to-end directly on raw MH-Weed16 captures using Ultralytics YOLO11 Nano (`yolo11n.pt`).

### Training Configuration
- **Hardware Acceleration**: Intel Arc XPU (`device='xpu'`).
- **Precision**: Automatic Mixed Precision (`amp=True`, float16) providing 2x acceleration.
- **Rectangular Batching (`rect=True`)**: Adapts to 16:9 1080p dimensions with minimal letterboxing padding.
- **Deterministic Disk Caching (`cache='disk'`)**: Fast batch streaming without RAM overflow.
- **Optimizer**: Explicit AdamW with learning rate 0.001 and weight decay 0.0005.
- **Fault-Tolerant Resumption**: Detects `last.pt` checkpoint automatically if interrupted.

In [ ]:
epochs = 50
batch = 16
imgsz = 1080
workers = 4
run_name = "mhweed16_yolo11n"

run_dir = ROOT_DIR / "runs" / "detect" / run_name
last_weights = run_dir / "weights" / "last.pt"
best_weights = run_dir / "weights" / "best.pt"

print("=" * 65)
print(f"Run Name:      {run_name}")
print(f"Device:        {device} ({device_name})")
print(f"Epochs:        {epochs} | Batch: {batch} | Image Size: {imgsz} | Workers: {workers}")
print(f"Precision:     Automatic Mixed Precision (amp=True)")
print(f"Batching:      Rectangular Training (rect=True)")
print(f"Caching:       Disk Caching (cache='disk')")
print(f"Optimizer:     AdamW")
print("=" * 65)

if last_weights.exists():
    print(f"Resuming training from checkpoint: {last_weights}")
    model = YOLO(str(last_weights))
    results = model.train(resume=True)
else:
    print("Starting fresh end-to-end training with yolo11n.pt...")
    model = YOLO("yolo11n.pt")
    
    results = model.train(
        data="mhweed16.yaml",
        epochs=epochs,
        batch=batch,
        imgsz=imgsz,
        rect=True,
        device=device,
        workers=workers,
        cache='disk',
        name=run_name,
        exist_ok=True,
        amp=True,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        degrees=15.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        patience=15
    )

print(f"Training complete. Best checkpoint saved at: {best_weights if best_weights.exists() else 'N/A'}")


## 9. Evaluation and Test Set Metrics
Evaluate detector metrics on held-out test split (`test.txt`). Computes mAP@0.50, mAP@0.50:0.95, precision, and recall.

In [ ]:
eval_weights = best_weights if best_weights.exists() else pathlib.Path("yolo11n.pt")
print(f"Evaluating checkpoint: {eval_weights}")
eval_model = YOLO(str(eval_weights))

metrics = eval_model.val(
    data="mhweed16.yaml",
    split="test",
    device=device,
    batch=batch,
    imgsz=imgsz,
    rect=True
)

print("\n" + "=" * 45)
print("           TEST SET EVALUATION METRICS       ")
print("=" * 45)
print(f"  mAP@0.50:       {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95:  {metrics.box.map:.4f}")
print(f"  Mean Precision: {metrics.box.mp:.4f}")
print(f"  Mean Recall:    {metrics.box.mr:.4f}")
print("=" * 45)


## 10. Slicing Aided Hyper Inference (SAHI) Visualization
Performs on-the-fly sliding window prediction (640x640 slices with 20% overlap) across full 1080p test images without downsampling or blur, and displays detections alongside ground truth annotations.

In [ ]:
with open("test.txt", 'r', encoding='utf-8') as f:
    test_image_paths = [pathlib.Path(line.strip()) for line in f if line.strip()]

selected_samples = random.sample(test_image_paths, min(3, len(test_image_paths)))

# Initialize SAHI AutoDetectionModel
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=str(eval_weights),
    confidence_threshold=0.25,
    device=device if device != 'xpu' else 'cpu'  # Fallback to CPU for slice postprocessing if needed
)

fig, axes = plt.subplots(len(selected_samples), 2, figsize=(16, 6 * len(selected_samples)))
if len(selected_samples) == 1:
    axes = np.expand_dims(axes, axis=0)

for idx, test_p in enumerate(selected_samples):
    raw = cv2.imread(str(test_p))
    gt_img = cv2.cvtColor(raw.copy(), cv2.COLOR_BGR2RGB)
    pred_img = cv2.cvtColor(raw.copy(), cv2.COLOR_BGR2RGB)
    h, w, _ = gt_img.shape
    
    # Render Ground Truth
    ann_p = LABELS_DIR / f"{test_p.stem}.txt"
    if ann_p.exists():
        with open(ann_p, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(gt_img, CLASS_NAMES.get(cid, str(cid)), (x1, max(20, y1 - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Execute SAHI Sliced Prediction (640x640 sliding window)
    sahi_result = get_sliced_prediction(
        str(test_p),
        sahi_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )
    
    for obj in sahi_result.object_prediction_list:
        bbox = obj.bbox.to_xyxy()
        x1, y1, x2, y2 = map(int, bbox)
        cat_name = obj.category.name
        score = obj.score.value
        cv2.rectangle(pred_img, (x1, y1), (x2, y2), (255, 50, 50), 2)
        cv2.putText(pred_img, f"{cat_name} {score:.2f}", (x1, max(20, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 50, 50), 2)

    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"Ground Truth ({test_p.name})", fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(pred_img)
    axes[idx, 1].set_title(f"SAHI 640x640 Sliced Inference ({len(sahi_result.object_prediction_list)} weeds)", fontsize=11, fontweight='bold')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()


## 11. Model Export (ONNX Format)

In [ ]:
if best_weights.exists():
    try:
        onnx_path = eval_model.export(format="onnx", dynamic=True)
        print(f"Model successfully exported to ONNX: {onnx_path}")
    except Exception as e:
        print(f"ONNX export notification: {e}")
else:
    print("No trained best_weights found to export. Skipping ONNX export.")
